In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken", version("tiktoken"))

torch version: 2.10.0
tiktoken 0.12.0


## Tokenizing text

In [2]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r") as f:
    raw_text = f.read()

print(f"total characters in the text: {len(raw_text)}")
print(f"first 100 characters: {raw_text[:100]}")

total characters in the text: 20479
first 100 characters: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [ ]:
import re

text = "hello, world, this is a test."
result = re.split(r'(\s)', text)
print(result)

['hello,', ' ', 'world,', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test.']


In [5]:
# splitting on whitespace, commas and periods
result = re.split(r'([,.]|\s)', text)
print(result)

['hello', ',', '', ' ', 'world', ',', '', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [6]:
# strip whitespace from each other and then filter out any empty strings
result = [item for item in result if item.strip()]
print(result)

['hello', ',', 'world', ',', 'this', 'is', 'a', 'test', '.']


In [7]:

text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [9]:
# applying the same to the full text
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(f"vocab size: {vocab_size}")

vocab size: 1130


In [13]:
vocab = {token: idx for idx, token in enumerate(all_words)}

# checking some vocab
for idx, item in enumerate(vocab):
    print(f"{idx}: {item}")
    if idx > 50:
        break

0: !
1: "
2: '
3: (
4: )
5: ,
6: --
7: .
8: :
9: ;
10: ?
11: A
12: Ah
13: Among
14: And
15: Are
16: Arrt
17: As
18: At
19: Be
20: Begin
21: Burlington
22: But
23: By
24: Carlo
25: Chicago
26: Claude
27: Come
28: Croft
29: Destroyed
30: Devonshire
31: Don
32: Dubarry
33: Emperors
34: Florence
35: For
36: Gallery
37: Gideon
38: Gisburn
39: Gisburns
40: Grafton
41: Greek
42: Grindle
43: Grindles
44: HAD
45: Had
46: Hang
47: Has
48: He
49: Her
50: Hermia
51: His


In [17]:
# building a simple tokenizer

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_id = vocab
        self.id_to_str  = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_id[token] for token in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join(self.id_to_str[id] for id in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
    
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [18]:
# decoding the ids back to text
decoded_text = tokenizer.decode(ids)
print(decoded_text)

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [35]:
# adding special context tokens
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab2 = {token: idx for idx, token in enumerate(all_tokens)}
print(len(vocab2.items()))

1132


In [36]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [37]:
tokenizerv2 = SimpleTokenizerV2(vocab2)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join([text1, text2])
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [38]:
tokenizerv2.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [39]:
print(tokenizerv2.decode(tokenizerv2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [40]:
# Using the BPE (Byte Pair Encoding) tokenizer from the tiktoken library
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [41]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [42]:
# Data sampling with a sliding window
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [45]:
# creating a dataset and dataloader that extract chunks from input text dataset
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length

        # Use a sliding window to chunk the book into overlapping segments
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [46]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [47]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
# Creating token embeddings
